In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Crude Oil Prices and Geopolitical Risk — 2010 to 2026

**Author:** Ahish Chandra  
**Dataset:** Daily Brent and WTI crude oil prices paired with 35 major geopolitical events, macro indicators (DXY, VIX), and the Caldara-Iacoviello Geopolitical Risk (GPR) Index.

---

### Research Questions
1. How do geopolitical shocks affect short-term oil prices, and how quickly do markets recover?
2. Can geopolitical risk and macro indicators predict oil price levels?
3. Which events caused the largest price dislocations — and which ones markets shrugged off?

### Methodology
- **Exploratory analysis** of price history, event timeline, and macro correlations
- **XGBoost regression** trained on lag features, macro indicators, and event flags
- **Event study analysis** measuring price impact and recovery windows (7, 30, 60, 90 days) around each geopolitical event

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Oil/oil_geopolitics_dataset_2010_2026.csv')
print(df.shape)
print(df.columns.tolist())
print(df.head())


(4047, 23)
['date', 'brent_price', 'wti_price', 'dxy_index', 'vix', 'gpr_index', 'brent_return', 'wti_return', 'brent_lag_1', 'brent_lag_3', 'brent_lag_7', 'wti_lag_1', 'wti_lag_3', 'wti_lag_7', 'brent_volatility_7d', 'brent_volatility_30d', 'wti_volatility_7d', 'wti_volatility_30d', 'brent_wti_spread', 'event_type', 'event_description', 'event_severity', 'event_flag']
         date  brent_price  wti_price  dxy_index        vix  gpr_index  \
0  2010-02-17    76.269997  77.330002  80.379997  21.719999  80.725357   
1  2010-02-18    77.779999  79.059998  80.400002  20.629999  80.725357   
2  2010-02-19    78.190002  79.809998  80.639999  20.020000  80.725357   
3  2010-02-22    78.610001  80.160004  80.510002  19.940001  80.725357   
4  2010-02-23    77.250000  78.860001  80.849998  21.370001  80.725357   

   brent_return  wti_return  brent_lag_1  brent_lag_3  ...  wti_lag_7  \
0      0.007796    0.004155    75.680000    73.050003  ...  71.190002   
1      0.019798    0.022372    76.269

In [ ]:
df.head()

,date,brent_price,wti_price,dxy_index,vix,gpr_index,brent_return,wti_return,brent_lag_1,brent_lag_3,...,wti_lag_7,brent_volatility_7d,brent_volatility_30d,wti_volatility_7d,wti_volatility_30d,brent_wti_spread,event_type,event_description,event_severity,event_flag
0,2010-02-17,76.269997,77.330002,80.379997,21.719999,80.725357,0.007796,0.004155,75.680000,73.050003,...,71.190002,0.014461,0.019679,0.016977,0.019280,-1.060005,none,none,0.0,0
1,2010-02-18,77.779999,79.059998,80.400002,20.629999,80.725357,0.019798,0.022372,76.269997,72.900002,...,71.889999,0.014387,0.020019,0.017366,0.019756,-1.279999,none,none,0.0,0
2,2010-02-19,78.190002,79.809998,80.639999,20.020000,80.725357,0.005271,0.009486,77.779999,75.680000,...,73.750000,0.013342,0.019796,0.016553,0.019559,-1.619995,none,none,0.0,0
3,2010-02-22,78.610001,80.160004,80.510002,19.940001,80.725357,0.005372,0.004385,78.190002,76.269997,...,74.519997,0.013366,0.019823,0.016772,0.019561,-1.550003,none,none,0.0,0
4,2010-02-23,77.250000,78.860001,80.849998,21.370001,80.725357,-0.017301,-0.016218,78.610001,77.779999,...,75.279999,0.017334,0.020045,0.019608,0.019756,-1.610001,none,none,0.0,0


In [ ]:
events=pd.read_csv('/content/drive/MyDrive/Oil/geopolitical_events_timeline.csv')
print(events.shape)
print(df.shape)


(35, 4)
(4047, 23)


In [ ]:
events.head()

,date,event_type,event_description,event_severity
0,2010-04-20,disaster,Deepwater Horizon oil spill,7
1,2011-02-15,war,Libyan Civil War begins,9
2,2011-03-19,war,NATO intervention in Libya,9
3,2012-01-23,sanctions,EU embargo on Iranian oil imports,8
4,2014-03-18,annexation,Russia annexes Crimea,8


In [ ]:
print("Main dataset shape:", df.shape)
print("\nMain columns:", df.columns.tolist())
print("\nEvents shape:", events.shape)
print("\nEvents columns:", events.columns.tolist())

Main dataset shape: (4047, 23)

Main columns: ['date', 'brent_price', 'wti_price', 'dxy_index', 'vix', 'gpr_index', 'brent_return', 'wti_return', 'brent_lag_1', 'brent_lag_3', 'brent_lag_7', 'wti_lag_1', 'wti_lag_3', 'wti_lag_7', 'brent_volatility_7d', 'brent_volatility_30d', 'wti_volatility_7d', 'wti_volatility_30d', 'brent_wti_spread', 'event_type', 'event_description', 'event_severity', 'event_flag']

Events shape: (35, 4)

Events columns: ['date', 'event_type', 'event_description', 'event_severity']


In [ ]:
events

,date,event_type,event_description,event_severity
0,2010-04-20,disaster,Deepwater Horizon oil spill,7
1,2011-02-15,war,Libyan Civil War begins,9
2,2011-03-19,war,NATO intervention in Libya,9
3,2012-01-23,sanctions,EU embargo on Iranian oil imports,8
4,2014-03-18,annexation,Russia annexes Crimea,8
5,2014-11-27,opec,OPEC maintains production despite falling prices,9
6,2015-03-26,war,Saudi-led intervention in Yemen,7
7,2016-01-16,sanctions,Iran nuclear sanctions lifted,6
8,2016-09-28,opec,OPEC Algiers agreement on production cuts,8
9,2016-11-30,opec,First OPEC+ production cut agreement,8


In [ ]:
df

,date,brent_price,wti_price,dxy_index,vix,gpr_index,brent_return,wti_return,brent_lag_1,brent_lag_3,...,wti_lag_7,brent_volatility_7d,brent_volatility_30d,wti_volatility_7d,wti_volatility_30d,brent_wti_spread,event_type,event_description,event_severity,event_flag
0,2010-02-17,76.269997,77.330002,80.379997,21.719999,80.725357,0.007796,0.004155,75.680000,73.050003,...,71.190002,0.014461,0.019679,0.016977,0.019280,-1.060005,none,none,0.0,0
1,2010-02-18,77.779999,79.059998,80.400002,20.629999,80.725357,0.019798,0.022372,76.269997,72.900002,...,71.889999,0.014387,0.020019,0.017366,0.019756,-1.279999,none,none,0.0,0
2,2010-02-19,78.190002,79.809998,80.639999,20.020000,80.725357,0.005271,0.009486,77.779999,75.680000,...,73.750000,0.013342,0.019796,0.016553,0.019559,-1.619995,none,none,0.0,0
3,2010-02-22,78.610001,80.160004,80.510002,19.940001,80.725357,0.005372,0.004385,78.190002,76.269997,...,74.519997,0.013366,0.019823,0.016772,0.019561,-1.550003,none,none,0.0,0
4,2010-02-23,77.250000,78.860001,80.849998,21.370001,80.725357,-0.017301,-0.016218,78.610001,77.779999,...,75.279999,0.017334,0.020045,0.019608,0.019756,-1.610001,none,none,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4042,2026-03-06,92.690002,90.900002,98.989998,29.490000,130.887665,0.085236,0.122084,85.410004,81.400002,...,65.419998,0.033662,0.029489,0.045250,0.034029,1.790001,none,none,0.0,0
4043,2026-03-09,98.959999,94.769997,99.180000,25.500000,130.887665,0.067645,0.042574,92.690002,81.400002,...,65.209999,0.029502,0.031034,0.039384,0.034316,4.190002,blockade,Closure of Strait of Hormuz shipping lane,10.0,1
4044,2026-03-10,87.800003,83.449997,98.830002,24.930000,130.887665,-0.112773,-0.119447,98.959999,85.410004,...,67.019997,0.068590,0.038639,0.077506,0.042092,4.350006,none,none,0.0,0
4045,2026-03-11,91.980003,87.250000,99.230003,24.230000,130.887665,0.047608,0.045536,87.800003,92.690002,...,71.230003,0.066619,0.039074,0.076723,0.042435,4.730003,none,none,0.0,0


In [ ]:
print("Main dataset shape:", df.shape)
print("\nMain columns:", df.columns.tolist())
print("\nEvents shape:", events.shape)
print("\nEvents columns:", events.columns.tolist())

Main dataset shape: (4047, 23)

Main columns: ['date', 'brent_price', 'wti_price', 'dxy_index', 'vix', 'gpr_index', 'brent_return', 'wti_return', 'brent_lag_1', 'brent_lag_3', 'brent_lag_7', 'wti_lag_1', 'wti_lag_3', 'wti_lag_7', 'brent_volatility_7d', 'brent_volatility_30d', 'wti_volatility_7d', 'wti_volatility_30d', 'brent_wti_spread', 'event_type', 'event_description', 'event_severity', 'event_flag']

Events shape: (35, 4)

Events columns: ['date', 'event_type', 'event_description', 'event_severity']


In [ ]:
import matplotlib.pyplot as plt

# Parse dates
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Check nulls
print("Nulls:\n", df.isnull().sum())
print("\nDate range:", df['date'].min(), "to", df['date'].max())
print("\nBasic stats:\n", df[['brent_price','wti_price','vix','gpr_index']].describe())


Nulls:
 date                    0
brent_price             0
wti_price               0
dxy_index               0
vix                     0
gpr_index               0
brent_return            0
wti_return              0
brent_lag_1             0
brent_lag_3             0
brent_lag_7             0
wti_lag_1               0
wti_lag_3               0
wti_lag_7               0
brent_volatility_7d     0
brent_volatility_30d    0
wti_volatility_7d       0
wti_volatility_30d      0
brent_wti_spread        0
event_type              0
event_description       0
event_severity          0
event_flag              0
dtype: int64

Date range: 2010-02-17 00:00:00 to 2026-03-12 00:00:00

Basic stats:
        brent_price    wti_price          vix    gpr_index
count  4047.000000  4047.000000  4047.000000  4047.000000
mean     77.623944    71.417546    18.393249   102.596421
std      23.529261    20.777323     6.853769    31.559190
min      19.330000   -37.630001     9.140000    58.420769
25%      60.785000  

In [ ]:
## Brent and WTI Price History with Geopolitical Events

The chart plots Brent and WTI crude oil prices from 2010 to 2026. Red vertical lines mark major geopolitical events — wars, sanctions, OPEC decisions, and supply disruptions.

1. **2020 COVID crash** was the single largest price collapse in the dataset. Global travel halted overnight, demand evaporated, and WTI turned negative for the first time in history — storage facilities were completely full, leaving no physical location to deliver barrels.

2. **VIX peaked at 82 in 2020**, the highest fear reading in the entire 16-year dataset, surpassing even the levels seen during the 2008 financial crisis.

3. **2016 oil glut** produced a similar price decline but for a structurally different reason. Saudi Arabia deliberately flooded the market to bankrupt US shale producers, and OPEC refused to cut supply even as prices fell below $30 per barrel.

4. **2022 GPR index spiked above 300**, the highest reading in the dataset by a wide margin. Russia's invasion of Ukraine severed Europe's energy relationship with Russia overnight — no other event in the dataset comes close to this reading on geopolitical risk.

#	 Brent and WTI Price History with Geopolitical Events
  Chart shows Brent and WTI crude oil prices from 2010 to 2026 with red lines marking major geopolitical events

	1. 2020 COVID crash was the biggest drop in the dataset. No travel, no demand, WTI went negative for the first time ever because storage was full

	2.	VIX hit 82 in 2020, highest panic reading in the entire dataset

	3. 2016 saw a similar price drop but different cause. Saudi Arabia flooded the market to kill US shale production and OPEC refused to cut supply

	4. 2022 GPR index spiked above 300, the highest in the dataset. Russia invaded Ukraine and Europe could no longer buy Russian oil. Nothing else comes close to this reading

In [ ]:
## Model Performance

1. **The model predicts WTI prices to within $1.25 per barrel on average.** When oil trades at $70–80 a barrel, a $1.25 error represents less than 2% of the actual price — strong accuracy for a commodity as volatile as crude oil.

2. **An R² score of 0.9553 means the model explains 95.5% of the variance in WTI prices**, leaving only 4.5% unexplained. This is an unusually high score, and the feature importance analysis in the next section explains exactly why — and why that should be treated with caution.

1. On average this model’s WTI price prediction is off by 1.25 per barrel. When oil trades at 70-80 a barrel, being wrong by $1.25 is less than 2% error.

2. 95.3% accuracy predicting model

In [ ]:
## Feature Importance Analysis

1. **WTI lag 1 (yesterday's price) dominates the model at 95% importance.** The model is essentially predicting that tomorrow's price will look a lot like today's — a phenomenon known as price persistence, or random-walk behaviour in financial markets.

2. **Brent lag 1 and WTI lag 3 follow at approximately 2% and 1.4% respectively** — both negligible in practice compared to the dominant lag-1 feature.

3. **This reveals the model is capturing price momentum, not genuine market intelligence.** High R² driven almost entirely by a single lag feature is a warning sign, not evidence of a smart model.

4. **Stress test — removing all lag features:** When retrained using only macro and geopolitical indicators (DXY, VIX, GPR, volatility, event flags), MAE jumped from $1.25 to $20.77 and R² went deeply negative.

5. **This confirms that geopolitical risk and macro indicators alone cannot predict absolute oil price levels** — but they are essential for explaining and flagging sudden price shocks during extreme events.

6. **The correct formulation of this problem is predicting daily returns, not raw prices.** Returns are stationary, free from lag-feature leakage, and represent a far harder and more meaningful challenge for real trading applications.

1. WTI lag 1 (yesterday’s price) dominates with 95% importance, meaning the model mostly predicts that tomorrow looks like today
2. Brent lag 1 and WTI lag 3 follow at around 2% and 1.4% respectively, basically negligible
3.	This reveals the model is capturing price momentum not real market intelligence
4.	When we removed all lag features and tested with only macro and geopolitical indicators, MAE jumped from $1.25 to $20.77 and R² went negative
5.	This proves geopolitical risk and macro indicators alone cannot predict oil price levels but they do explain sudden price shocks
6. A better model would predict daily returns instead of raw prices, which is a harder and more meaningful problem

In [ ]:
features_no_lag = ['dxy_index','vix','gpr_index',
                   'brent_volatility_7d','wti_volatility_7d',
                   'brent_wti_spread','event_flag','event_severity']

X2 = df[features_no_lag]

X_train2, X_test2 = X2[:split], X2[split:]

model2 = XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
model2.fit(X_train2, y_train)

preds2 = model2.predict(X_test2)
mae2 = mean_absolute_error(y_test, preds2)
r2_2 = r2_score(y_test, preds2)

print(f"MAE without lag features: ${mae2:.2f}")
print(f"R² without lag features: {r2_2:.4f}")


MAE without lag features: $20.77
R² without lag features: -8.3661


In [ ]:
## XGBoost: Actual vs Predicted WTI Price

1. **The predicted line tracks actual WTI prices almost perfectly across three years of unseen test data**, demonstrating strong out-of-sample generalisation.

2. **At $1.25 MAE, the model is off by less than 2% on average** on a commodity that traded between $60 and $95 per barrel throughout the test period.

3. **The only visible prediction errors occur at sharp peaks and troughs** — the most unpredictable moments in any market, and the exact points where no model trained on historical data alone can be expected to perform reliably.

4. **Stress test confirmation:** Removing lag features and relying solely on geopolitical and macro indicators caused performance to collapse — MAE rose to $20.77 and R² turned sharply negative.

5. **Conclusion:** Short-term price momentum drives day-to-day oil prices. Geopolitical risk is best used as a risk flag and event classifier during extreme market dislocations, not as a standalone price-level predictor.

	1. The predicted line follows actual WTI prices almost perfectly across 3 years of unseen test data
	2. The model achieves $1.25 MAE meaning on average it is only off by $1.25 per barrel on a commodity trading between $60 and $95
	3. The only visible errors occur at sharp peaks and troughs, which are the hardest moments to predict in any market
	3. When I stress tested the model by removing lag features and using only geopolitical and macro indicators, performance collapsed completely with MAE jumping to $20.77
	4. This confirms that short term price momentum drives day to day oil prices while geopolitical risk is better used as a risk flag during extreme events rather than a standalone price predictor

In [ ]:
# Merge events with main df on date
df_events = df[df['event_flag'] == 1][['date','wti_price','brent_price','gpr_index','vix','event_description','event_severity']].copy()

# Calculate price 30 days before and 30 days after each event
results = []

for _, event in events.iterrows():
    event_date = event['date']

    # Get index of event date in main df
    idx = df[df['date'] == event_date].index
    if len(idx) == 0:
        continue
    idx = idx[0]

    # Price on event day
    price_on_day = df.loc[idx, 'wti_price']

    # Price 30 days before
    before_idx = max(0, idx - 30)
    price_before = df.loc[before_idx, 'wti_price']

    # Price 30 days after
    after_idx = min(len(df)-1, idx + 30)
    price_after = df.loc[after_idx, 'wti_price']

    # Calculate impact and recovery
    impact = price_on_day - price_before
    recovery = price_after - price_on_day
    pct_impact = (impact / price_before) * 100

    results.append({
        'date': event_date,
        'event': event['event_description'],
        'severity': event['event_severity'],
        'price_before': round(price_before, 2),
        'price_on_day': round(price_on_day, 2),
        'price_after_30d': round(price_after, 2),
        'impact_usd': round(impact, 2),
        'impact_pct': round(pct_impact, 2),
        'recovery_usd': round(recovery, 2)
    })

results_df = pd.DataFrame(results).sort_values('impact_pct')
print(results_df[['date','event','impact_pct','recovery_usd']].to_string())


         date                                              event  impact_pct  recovery_usd
10 2020-04-20                     WTI crude prices turn negative     -191.16         74.44
16 2022-12-05                        G7 price cap on Russian oil       -9.55          3.40
1  2011-02-15                            Libyan Civil War begins       -7.90         19.95
22 2025-10-28            World Bank warns of major 2026 oil glut       -6.77         -1.69
17 2024-01-12                  Red Sea shipping attacks escalate       -6.65          6.19
19 2024-10-01                    Expanded Red Sea tanker attacks       -6.10         -1.71
18 2024-08-26  Libya eastern administration halts oil production       -5.48         -3.85
7  2017-06-05                     Qatar diplomatic crisis begins       -4.47         -1.00
20 2025-02-20                 Middle East tanker security crisis       -2.26         -5.62
6  2016-11-30               First OPEC+ production cut agreement       -1.69          2.93

In [ ]:
## Price Impact Chart (Left Panel)

1. **The April 2020 WTI negative price event produced the largest price dislocation in the dataset at −191%** — a move without precedent in commodity market history, driven by a simultaneous collapse in demand and the complete exhaustion of available storage capacity.

2. **The G7 price cap on Russian oil and the Libyan Civil War also caused significant price drops**, as supply uncertainty rattled markets and investors priced in potential volume shortfalls before the actual disruptions materialised.

3. **The largest positive price spikes were caused by the Strait of Hormuz closure, the US ban on Russian oil imports, the global energy crisis, and Russia's invasion of Ukraine.**

4. **These positive spikes share a common structural driver** — each event either threatened a critical supply route or simultaneously removed large volumes of oil from global markets, forcing prices sharply higher in a short period of time.

## 30-Day Recovery Chart (Right Panel)

1. **Markets recovered fastest after WTI turned negative and after the Libyan Civil War** — both cases where the initial panic reaction far exceeded the actual long-term supply damage, leading to a sharp mean reversion once markets reassessed the fundamentals.

2. **The weakest recoveries followed the US ban on Russian oil imports, the EU oil embargo on Russia, the Deepwater Horizon spill, and the assassination of Iranian General Qasem Soleimani.**

3. **These events produced lasting structural consequences** that markets could not quickly absorb — supply routes were permanently rerouted, sanctions remained in force for years, and the underlying disruptions were real and sustained rather than psychological.

4. **The pattern across both charts reveals a consistent dynamic:** markets tend to overreact to short-term shocks but struggle to recover when the underlying supply disruption is genuine and sustained rather than speculative.

# 30 Day Recovery Chart (Right)
1. Markets recovered fastest after WTI went negative and after the Libyan Civil War, both cases where the initial panic was worse than the actual supply damage
2. The worst recoveries happened after the US ban on Russian oil imports, EU oil embargo on Russia, Deepwater Horizon oil spill, and the assassination of Iranian General Qasem Soleimani
3. These events had lasting structural consequences that markets could not quickly absorb, meaning prices stayed disrupted well beyond 30 days
4.	The pattern across both charts shows that markets overreact to short term shocks but struggle to recover when the underlying supply disruption is real and sustained

In [ ]:
## Correlation Matrix: Oil Prices vs Macro and Geopolitical Indicators

1. **Brent and WTI prices are almost perfectly correlated at 0.98.** Both benchmarks track the same underlying global commodity supply-demand balance, differing mainly in delivery location and pipeline versus seaborne transportation costs.

2. **Event flag and event severity are correlated at 0.99**, which is expected — they are two representations of the same underlying variable: the presence and magnitude of a geopolitical event on a given day.

3. **The US Dollar Index (DXY) is inversely correlated with Brent (−0.47) and WTI (−0.54).** Since oil is priced in dollars globally, a stronger dollar effectively makes oil more expensive for foreign buyers, suppressing demand and exerting downward pressure on prices.

4. **30-day WTI volatility is negatively correlated with price at −0.27**, meaning elevated volatility periods tend to coincide with lower and declining prices — consistent with the crash periods in 2016 and 2020.

5. **The GPR index shows only 0.12 correlation with WTI prices** — confirming that geopolitical risk explains sudden price shocks but does not sustain elevated price levels over time once the immediate uncertainty passes.

6. **Brent-WTI spread correlates at 0.60 with Brent price**, meaning the premium between the two benchmarks widens when Brent is expensive — typically during supply disruptions affecting seaborne crude routes such as the Strait of Hormuz or Suez Canal, which impact Brent more directly than landlocked WTI.

#	Plot: Correlation Matrix - Oil Prices vs Macro and Geopolitical Indicators
  1.	Brent and WTI prices are almost perfectly correlated at 0.98, they move together because they are both benchmarks for the same global commodity

  2. Event flag and event severity are correlated at 0.99, which makes sense as they are essentially measuring the same thing

  3. The US Dollar index is inversely correlated with both Brent and WTI at -0.47 and -0.54, meaning when the dollar strengthens oil gets cheaper and vice versa

  4. WTI 30 day volatility is negatively correlated with price at -0.27, meaning higher volatility periods tend to coincide with lower and falling prices

  5. Surprisingly GPR index shows only 0.12 correlation with WTI prices, confirming that geopolitical risk explains sudden price shocks but does not drive sustained price levels

  6. Brent WTI spread correlates at 0.60 with Brent price, meaning the gap between the two benchmarks widens when Brent is expensive, typically during supply disruptions affecting seaborne crude

In [ ]:
## Average Market Recovery Across Different Time Windows

1. **The highest average recovery occurs within the first 7 days at $3.62** — evidence that markets systematically overreact to geopolitical headlines immediately after an event, then partially correct once the actual supply impact becomes clearer.

2. **Recovery drops to $2.08 at 30 days** as markets continue to digest the structural consequences of each event — the initial snap-back is partially reversed as longer-term reality sets in.

3. **By 60 and 90 days, recovery stabilises at $2.93 and $2.99 respectively**, suggesting that most geopolitical shocks are fully priced into oil markets within approximately two months.

4. **The gap between 7-day and 90-day recovery has a clear practical implication:** traders who hold through the initial panic are ultimately rewarded on average, while those who react immediately to breaking news tend to buy or sell at the worst possible moment.

5. **For energy market practitioners, the 7-day overreaction window represents a mean-reversion opportunity**, while the 60-day stabilisation point signals when a new post-event price equilibrium has typically been established — a useful framework for positioning around high-severity geopolitical events.

## Next Steps

1. **Predict daily returns instead of raw prices** — a stationary, leak-free formulation that eliminates lag-feature dominance and represents a harder, more meaningful problem for real trading applications.

2. **Implement proper event study methodology** with pre-event and post-event control windows to isolate the true causal price impact of each geopolitical event from concurrent market noise.

3. **Segment the analysis across market regimes** — pre-COVID (2010–2019), COVID (2020–2021), and post-COVID (2022–present) — since volatility structure, OPEC discipline, and macro correlations differ significantly across these periods and pooling them together may mask regime-specific dynamics.

# Next Steps
	1.	Build a model predicting daily returns instead of raw prices, a harder and more meaningful problem for real trading applications
	2. Implement proper event study methodology with control windows to isolate true geopolitical impact from market noise
	3. Segment analysis across pre COVID, COVID, and post COVID periods as market behavior differs significantly across these regimes